In [1]:
from chemplus import prep
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.warning')
RDLogger.DisableLog('rdApp.error')
pd.set_option("display.max_colwidth", None)

# Set parameters for preparation

In [2]:
smiles_file_path = "../../../pre-trained/novel_molecules_2026-08-06_22-39.csv"
smiles_column = "SMILES"
new_name_column = "Mol ID"
library_name = "prepared"

output_bad_csv = "../../../pre-trained/prepared_bad_smiles.csv"
output_passed_csv = "../../../pre-trained/prepared_passed.csv"
output_discarded_csv = "../../../pre-trained/prepared_discarded.csv"

# Prepare and save csv files

This script checks whether the molecules in SMILES are suitable for use in our molecular modeling methods and prepares them. 

The preparation of molecules consists of the following steps:
1. Checking whether the molecule's structure is determined 
2. Removal of cations
3. Splitting mixtures into separate fragments
3. Checking fragments for typical atomic isotopes
4. Checking fragments for inorganics
5. Checking for atom types not supported by molecular modeling software and atom types not characteristic of drugs

In [3]:
if smiles_file_path.endswith(".csv"):
    df = pd.read_csv(smiles_file_path)
elif smiles_file_path.endswith(".xlsx"):
    df = pd.read_excel(smiles_file_path)
else:
    raise Exception("Unknown smiles file format")
print(f"Number of SMILES entries: {df.shape[0]}")
df["Prep results"] = df[smiles_column].apply(Chem.MolFromSmiles)
print(f"Number of bad SMILES: {df['Prep results'].isna().sum()}")
df[df['Prep results'].isna()].drop(columns="Prep results").to_csv(output_bad_csv, index=False)
df = df[df["Prep results"].notna()]
df["Prep results"] = df["Prep results"].apply(prep.prep_mol)
df["Prep results"] = df["Prep results"].apply(lambda x: [(frag, log,  f"frag-{idx+1}") for idx, (frag,  log) in enumerate(x)])
df = df.explode("Prep results", ignore_index=False)
print(f"Number of Fragments: {df.shape[0]}")
df["Fragment"], df["Result"], df["Fragment number"] = tuple(zip(*df["Prep results"]))
df.drop(columns="Prep results", inplace=True)
df["Fragment SMILES"] = df["Fragment"].apply(Chem.MolToSmiles)
df.reset_index(inplace=True, drop=True)
df = df.sort_values(by="Fragment number")
df = df[~df["Fragment SMILES"].duplicated()]
print(f"Number of unique Fragments: {df.shape[0]}")
df = df.sort_index()
print(f"Number of discarded Fragments to save: {df[df['Result'] != 'OK'].shape[0]}")
df[df["Result"] != "OK"].drop(columns="Fragment").to_csv(output_discarded_csv, index=False)
df_ok = df[df['Result'] == 'OK'].copy(deep=True)
print(f"Number of passed Fragments to save: {df_ok.shape[0]}")
df_ok[new_name_column] = [f"{library_name}-{i+1}" for i in range(df_ok.shape[0])]
df_ok.drop(columns=["Fragment", "Result"]).to_csv(output_passed_csv, index=False)

Number of SMILES entries: 92389
Number of bad SMILES: 0
Number of Fragments: 92400
Number of unique Fragments: 92391
Number of discarded Fragments to save: 3862
Number of passed Fragments to save: 88529
